# AISC DeepFake — Equal Soft Voting Region Fusion

Bu notebook, **aynı model ailesinin** göz, kaş ve ağız bölgesel tahminlerini
**Equal Soft Voting** ile birleştirir.

## Model families
1. Swin V2 Tiny
2. EfficientNet-B0
3. Swin V2 Tiny + Texture Fusion

## Fusion rule

\[
p_{fusion} = \frac{p_{eye} + p_{brow} + p_{mouth}}{3}
\]

## Scientific constraints

- Göz, kaş ve ağız CSV'leri satır sırasına göre eşleştirilmez.
- Yalnızca aynı upstream frame kimliğine sahip ortak örnekler kullanılır.
- Test seti threshold seçimi veya model seçimi için kullanılmaz.
- Equal Soft Voting için karar eşiği **önceden tanımlı 0.50** olarak tutulur.
- Her model ailesinde Eye-only, Brow-only, Mouth-only ve Fusion sonuçları
  **aynı ortak frame seti** üzerinde hesaplanır.
- Kaynak prediction/ROI dosyaları yalnızca okunur; bu notebook onları silmez,
  taşımaz veya üzerine yazmaz.
- Bir frame'de birden fazla ROI/yüz tahmini varsa frame olasılığı görünür biçimde
  `mean` ile toplanır ve ROI sayısı audit çıktısında saklanır.

## Standards implemented

Notebook; görünürlük/doğrulanabilirlik, seed/reproducibility, veri sızıntısı
önleme, fail-fast quality gates, atomik çıktı yazımı ve yüksek çözünürlüklü
İngilizce grafik üretimi prensipleriyle düzenlenmiştir.

Üretilen grafikler hem **PNG (600 DPI)** hem **SVG** olarak kaydedilir.

In [1]:
# ============================================================
# 1) COLAB + CONFIG — DÜZELTİLMİŞ
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import re
import math
import warnings
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "AISC DeepFake Çalışmaları/Deney 1/"
    "Kader/Deney 1/Sonuçlar/Fusion_Experiments"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# ANA SONUÇ KLASÖRLERİ
# ============================================================

SEARCH_ROOTS = {

    "eye": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Kader/Deney 1/Sonuçlar"
    ),

    "brow": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Nazlıcan/Deney 1/Sonuçlar"
    ),

    "mouth": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Dilara/Deney 1/Sonuçlar"
    ),
}


# ============================================================
# MODEL AİLELERİ
# ============================================================

MODEL_FAMILIES = {

    # --------------------------------------------------------
    # 1) SWIN V2 TINY
    # --------------------------------------------------------
    "swinv2_tiny": {

        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260807_1031_eye_swinv2_tiny_seed42"
            / "predictions"
            / "test_frame_predictions.csv"
        ),

        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "Kas_SwinV2_Tiny_Detayli_Sonuc_pdf"
            / "predictions"
            / "test_frame_predictions.csv"
        ),

        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "20260807_2235_mouth_swinv2_tiny_seed42"
            / "predictions"
            / "test_frame_predictions.csv"
        ),

        "eye_val": None,
        "brow_val": None,
        "mouth_val": None,
    },


    # --------------------------------------------------------
    # 2) EFFICIENTNET-B0
    # --------------------------------------------------------
    "efficientnet_b0": {

        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260808_0803_eye_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),

        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "20260808_1248_eyebrow_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),

        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "20260808_1257_mouth_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),

        "eye_val": None,
        "brow_val": None,
        "mouth_val": None,
    },


    # --------------------------------------------------------
    # 3) SWIN V2 TINY + TEXTURE FUSION
    # --------------------------------------------------------
    "swinv2_texture": {

    "eye_test": (
        SEARCH_ROOTS["eye"]
        / "20260806_1748_eye_swinv2_texturefusion_seed42"
        / "full"
        / "predictions"
        / "test_predictions.csv"
    ),

    "brow_test": (
        SEARCH_ROOTS["brow"]
        / "Swin V2-Tiny + LBP + GLCM + Gabor + Wavelet Fusion"
        / "predictions"
        / "test_predictions_frame_level.csv"
    ),

    "mouth_test": (
        SEARCH_ROOTS["mouth"]
        / "SwinV2_TextureFusion_Mouth"
        / "20260807_1550_mouth_swinv2_texturefusion_seed42"
        / "full"
        / "predictions"
        / "test_predictions.csv"
    ),

    "eye_val": None,
    "brow_val": None,
    "mouth_val": None,
},
}


# ============================================================
# PATH QUALITY GATE
# ============================================================

print("\n" + "=" * 90)
print("MODEL DOSYASI KONTROLÜ")
print("=" * 90)

all_test_paths_ok = True

for family, cfg in MODEL_FAMILIES.items():

    print(f"\n[{family}]")

    for region in ("eye", "brow", "mouth"):

        key = f"{region}_test"
        path = cfg[key]

        if path is None:
            status = "❌ NONE"
            all_test_paths_ok = False

        elif Path(path).is_file():
            status = "✅ BULUNDU"

        else:
            status = "❌ BULUNAMADI"
            all_test_paths_ok = False

        print(
            f"{region.upper():5s} | "
            f"{status} | "
            f"{path}"
        )


# ============================================================
# ÖZET
# ============================================================

print("\n" + "=" * 90)

if all_test_paths_ok:

    print("✅ TÜM TEST PREDICTION DOSYALARI BULUNDU.")
    print("Fusion aşamasına geçilebilir.")

else:

    print("⚠️ EN AZ BİR TEST DOSYASI BULUNAMADI.")
    print(
        "Yukarıdaki ❌ olan yolu kontrol et. "
        "Fusion hücresini henüz çalıştırma."
    )

print("=" * 90)


# ============================================================
# CONFIG ÖZETİ
# ============================================================

print("\nCONFIG:\n")

print(
    json.dumps(
        {
            family: {
                k: str(v) if v is not None else None
                for k, v in cfg.items()
            }
            for family, cfg in MODEL_FAMILIES.items()
        },
        indent=2,
        ensure_ascii=False,
    )
)

Mounted at /content/drive

MODEL DOSYASI KONTROLÜ

[swinv2_tiny]
EYE   | ✅ BULUNDU | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/20260807_1031_eye_swinv2_tiny_seed42/predictions/test_frame_predictions.csv
BROW  | ✅ BULUNDU | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/Sonuçlar/Kas_SwinV2_Tiny_Detayli_Sonuc_pdf/predictions/test_frame_predictions.csv
MOUTH | ✅ BULUNDU | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Dilara/Deney 1/Sonuçlar/20260807_2235_mouth_swinv2_tiny_seed42/predictions/test_frame_predictions.csv

[efficientnet_b0]
EYE   | ✅ BULUNDU | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/20260808_0803_eye_efficientnet_b0_seed42/predictions/test_predictions.csv
BROW  | ✅ BULUNDU | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/Sonuçlar/20260808_1248_eyebrow_efficientnet_b0_seed42/predictions/test_predictions.csv
MOUTH | ✅ BULUNDU | /content/d

In [8]:
# ============================================================
# 2) REPRODUCIBILITY + RUN METADATA + OUTPUT SAFETY
# ============================================================

import os
import platform
import sys
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version

import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

METHOD_NAME = "01_equal_soft_voting"
DECISION_THRESHOLD = 0.50
FRAME_AGGREGATION = "mean"
FIGURE_DPI = 600
MIN_FIGURE_SHORT_EDGE_PX = 600

RUN_ID = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%d_%H%M%S")
    + "_region_equal_soft_voting_seed42"
)

RUN_DIR = OUTPUT_ROOT / METHOD_NAME / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

print(f"Run ID     : {RUN_ID}")
print(f"Run output : {RUN_DIR}")


def package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not_installed"


ENVIRONMENT = {
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "scikit_learn": package_version("scikit-learn"),
    "pillow": package_version("Pillow"),
    "seed": SEED,
    "method": METHOD_NAME,
    "decision_threshold": DECISION_THRESHOLD,
    "frame_aggregation": FRAME_AGGREGATION,
    "figure_dpi": FIGURE_DPI,
}

Run ID     : 20260809_162333_region_equal_soft_voting_seed42
Run output : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/01_equal_soft_voting/20260809_162333_region_equal_soft_voting_seed42


In [9]:
# ============================================================
# 3) ATOMIC I/O HELPERS
# ============================================================

def atomic_write_json(payload, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target.with_suffix(target.suffix + ".tmp")

    with temp_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())

    with temp_path.open("r", encoding="utf-8") as f:
        json.load(f)

    os.replace(temp_path, target)


def atomic_write_csv(df, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target.with_suffix(target.suffix + ".tmp")

    df.to_csv(temp_path, index=False)

    verification = pd.read_csv(temp_path)
    if len(verification) != len(df):
        raise RuntimeError(
            f"Atomic CSV verification failed for {target}: "
            f"{len(df)} rows expected, {len(verification)} rows read back."
        )

    os.replace(temp_path, target)


def save_figure(fig, target_stem):
    target_stem = Path(target_stem)
    target_stem.parent.mkdir(parents=True, exist_ok=True)

    png_path = target_stem.with_suffix(".png")
    svg_path = target_stem.with_suffix(".svg")
    temp_png = png_path.with_suffix(".png.tmp")
    temp_svg = svg_path.with_suffix(".svg.tmp")

    fig.savefig(
        temp_png,
        format="png",
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )
    fig.savefig(
        temp_svg,
        format="svg",
        bbox_inches="tight",
    )

    with Image.open(temp_png) as img:
        width, height = img.size
        if min(width, height) < MIN_FIGURE_SHORT_EDGE_PX:
            raise RuntimeError(
                f"Figure resolution is insufficient: {img.size} for {png_path}"
            )

    os.replace(temp_png, png_path)
    os.replace(temp_svg, svg_path)
    plt.close(fig)

    return png_path, svg_path


atomic_write_json(ENVIRONMENT, RUN_DIR / "environment.json")

In [10]:
# ============================================================
# 4) CSV SCHEMA + FRAME KEY + ALIGNMENT HELPERS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

LABEL_CANDIDATES = [
    "true_label",
    "label",
    "label_int",
    "target",
    "y_true",
    "ground_truth",
    "class_id",
    "true_class",
]

PROB_CANDIDATES = [
    "fake_probability",
    "prob_fake",
    "probability_fake",
    "probability",
    "prob",
    "y_score",
    "score",
    "fake_prob",
    "prediction_probability",
]

KEY_CANDIDATES = [
    "source_frame",
    "relative_frame_path",
    "frame_path",
    "original_frame",
    "image_path",
    "path",
    "sample_id",
    "frame_stem",
]


def first_existing(columns, candidates):
    lower_to_original = {
        str(column).lower(): column
        for column in columns
    }

    for candidate in candidates:
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]

    return None


def normalize_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, np.integer, float, np.floating)):
        return int(float(value) >= 0.5)

    normalized = str(value).strip().lower()

    fake_values = {
        "1",
        "fake",
        "deepfake",
        "manipulated",
        "sahte",
        "f",
    }
    real_values = {
        "0",
        "real",
        "original",
        "genuine",
        "gerçek",
        "r",
    }

    if normalized in fake_values:
        return 1

    if normalized in real_values:
        return 0

    try:
        return int(float(normalized) >= 0.5)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"Label could not be normalized: {value!r}"
        ) from exc


def canonical_frame_key(value):
    """
    Convert region-specific ROI filenames to one upstream frame key.

    Examples
    --------
    eye   : fake_test_00000__face_00.jpg
    brow  : fake_test_00000.jpg
    mouth : fake_test_fake_test_00000_face00.png

    result: fake_test_00000

    Unknown formats are NOT guessed. They return None and fail a quality gate.
    """

    if pd.isna(value):
        return None

    text = str(value).replace("\\", "/").strip().lower()
    filename = text.split("/")[-1]

    filename = re.sub(
        r"\.(jpg|jpeg|png|bmp|webp|npy)$",
        "",
        filename,
        flags=re.IGNORECASE,
    )

    matches = re.findall(
        r"(real|fake)_(train|test|val|validation)_(\d+)",
        filename,
        flags=re.IGNORECASE,
    )

    if not matches:
        return None

    label, split, frame_number = matches[-1]
    split = split.lower()

    if split == "validation":
        split = "val"

    frame_number = str(int(frame_number)).zfill(5)

    return f"{label.lower()}_{split}_{frame_number}"


# ============================================================
# ROBUST PREDICTION LOADER
# ============================================================

SOURCE_KEY_CANDIDATES = [
    "source_frame",
    "relative_frame_path",
    "input_path",
    "input_relative_path",
    "original_frame",
    "orijinal_yol",
    "frame_stem",
    "image_path",
    "path",
]


def find_companion_metadata(prediction_path: Path):
    """
    Prediction CSV'nin ait olduğu deney klasöründe prediction -> original
    frame eşlemesini içeren metadata dosyasını arar.

    Mevcut veri veya prediction dosyalarına hiçbir şekilde yazmaz.
    """

    prediction_path = Path(prediction_path)

    # Örnek:
    # experiment/
    # ├── predictions/test_predictions.csv
    # └── artifacts/eligible_metadata.csv

    experiment_root = prediction_path.parent.parent

    metadata_candidates = [
        experiment_root / "artifacts" / "eligible_metadata.csv",
        experiment_root / "artifacts" / "eligible_metadata_before_cache.csv",
        experiment_root / "artifacts" / "metadata_used.csv",
        experiment_root / "metadata_used.csv",
        experiment_root / "eligible_metadata.csv",
    ]

    existing = [
        path
        for path in metadata_candidates
        if path.is_file()
    ]

    if not existing:
        return None

    # Öncelik sırası yukarıdaki listeye göre korunur.
    return existing[0]


def build_key_from_metadata(
    prediction_df: pd.DataFrame,
    prediction_path: Path,
    region_name: str,
):
    """
    Cache path gerçek upstream frame bilgisini taşımıyorsa:
        prediction.sample_id
             ↓
        companion metadata
             ↓
        source_frame / relative_frame_path / frame_stem
             ↓
        fusion_key
    """

    if "sample_id" not in prediction_df.columns:
        raise ValueError(
            f"{region_name}: prediction path'leri upstream frame kimliği "
            "içermiyor ve sample_id kolonu da bulunmuyor."
        )

    metadata_path = find_companion_metadata(
        prediction_path
    )

    if metadata_path is None:
        raise FileNotFoundError(
            f"{region_name}: cache prediction tespit edildi ancak "
            f"companion metadata bulunamadı.\n"
            f"Prediction: {prediction_path}"
        )

    print(
        f"{region_name.upper()} | "
        f"Companion metadata: {metadata_path}"
    )

    metadata = pd.read_csv(
        metadata_path
    )

    if "sample_id" not in metadata.columns:
        raise ValueError(
            f"{region_name}: companion metadata içinde "
            "sample_id kolonu yok."
        )

    source_column = None

    for candidate in SOURCE_KEY_CANDIDATES:

        if candidate not in metadata.columns:
            continue

        test_keys = (
            metadata[candidate]
            .map(canonical_frame_key)
        )

        valid_fraction = (
            test_keys.notna().mean()
        )

        if valid_fraction >= 0.95:
            source_column = candidate
            break

    if source_column is None:
        raise ValueError(
            f"{region_name}: companion metadata içerisinde "
            "upstream frame anahtarı üretilebilecek kolon bulunamadı.\n"
            f"Metadata columns: {list(metadata.columns)}"
        )

    print(
        f"{region_name.upper()} | "
        f"Metadata source column: {source_column}"
    )

    mapping = (
        metadata[
            [
                "sample_id",
                source_column,
            ]
        ]
        .copy()
    )

    mapping["sample_id"] = (
        mapping["sample_id"]
        .astype(str)
        .str.strip()
    )

    mapping["fusion_key"] = (
        mapping[source_column]
        .map(canonical_frame_key)
    )

    mapping = (
        mapping.dropna(
            subset=["fusion_key"]
        )
        .drop_duplicates(
            subset=["sample_id"]
        )
    )

    work = prediction_df.copy()

    work["sample_id"] = (
        work["sample_id"]
        .astype(str)
        .str.strip()
    )

    merged = work.merge(
        mapping[
            [
                "sample_id",
                "fusion_key",
            ]
        ],
        on="sample_id",
        how="left",
        validate="many_to_one",
    )

    missing = int(
        merged["fusion_key"]
        .isna()
        .sum()
    )

    if missing:

        examples = (
            merged.loc[
                merged["fusion_key"].isna(),
                "sample_id",
            ]
            .head(10)
            .tolist()
        )

        raise ValueError(
            f"{region_name}: {missing} prediction sample_id "
            "metadata ile eşleşmedi.\n"
            f"Examples: {examples}"
        )

    return merged["fusion_key"], {
        "key_resolution": "companion_metadata",
        "metadata_path": str(metadata_path),
        "metadata_source_column": str(source_column),
    }


def resolve_fusion_key(
    raw: pd.DataFrame,
    prediction_path: Path,
    region_name: str,
):
    """
    1) Önce prediction CSV içindeki gerçek frame kolonlarını dener.
    2) Eğer path geçici cache ise sample_id -> metadata -> source frame
       zincirini kullanır.
    """

    # --------------------------------------------------------
    # 1) Prediction CSV içinden doğrudan çözmeyi dene
    # --------------------------------------------------------

    direct_candidates = [
        "source_frame",
        "relative_frame_path",
        "frame_path",
        "original_frame",
        "image_path",
        "path",
        "frame_stem",
    ]

    for column in direct_candidates:

        if column not in raw.columns:
            continue

        keys = (
            raw[column]
            .map(canonical_frame_key)
        )

        valid_fraction = float(
            keys.notna().mean()
        )

        # Tam veya neredeyse tam çözülebiliyorsa bunu kullan.
        if valid_fraction >= 0.95:

            print(
                f"{region_name.upper()} | "
                f"Fusion key source: prediction CSV -> {column}"
            )

            return keys, {
                "key_resolution": "prediction_column",
                "key_column": str(column),
            }


    # --------------------------------------------------------
    # 2) Cache/path hash tespit edildi:
    #    sample_id üzerinden metadata'ya dön
    # --------------------------------------------------------

    print(
        f"{region_name.upper()} | "
        "Direct frame key unavailable. "
        "Using sample_id -> companion metadata mapping."
    )

    return build_key_from_metadata(
        prediction_df=raw,
        prediction_path=prediction_path,
        region_name=region_name,
    )


# ============================================================
# ROBUST PREDICTION LOADER
# ============================================================

SOURCE_KEY_CANDIDATES = [
    "source_frame",
    "relative_frame_path",
    "input_path",
    "input_relative_path",
    "original_frame",
    "orijinal_yol",
    "frame_stem",
    "image_path",
    "path",
]


def find_companion_metadata(prediction_path: Path):
    """
    Prediction CSV'nin ait olduğu deney klasöründe prediction -> original
    frame eşlemesini içeren metadata dosyasını arar.

    Mevcut veri veya prediction dosyalarına hiçbir şekilde yazmaz.
    """

    prediction_path = Path(prediction_path)

    # Örnek:
    # experiment/
    # ├── predictions/test_predictions.csv
    # └── artifacts/eligible_metadata.csv

    experiment_root = prediction_path.parent.parent

    metadata_candidates = [
        experiment_root / "artifacts" / "eligible_metadata.csv",
        experiment_root / "artifacts" / "eligible_metadata_before_cache.csv",
        experiment_root / "artifacts" / "metadata_used.csv",
        experiment_root / "metadata_used.csv",
        experiment_root / "eligible_metadata.csv",
    ]

    existing = [
        path
        for path in metadata_candidates
        if path.is_file()
    ]

    if not existing:
        return None

    # Öncelik sırası yukarıdaki listeye göre korunur.
    return existing[0]


def build_key_from_metadata(
    prediction_df: pd.DataFrame,
    prediction_path: Path,
    region_name: str,
):
    """
    Cache path gerçek upstream frame bilgisini taşımıyorsa:
        prediction.sample_id
             ↓
        companion metadata
             ↓
        source_frame / relative_frame_path / frame_stem
             ↓
        fusion_key
    """

    if "sample_id" not in prediction_df.columns:
        raise ValueError(
            f"{region_name}: prediction path'leri upstream frame kimliği "
            "içermiyor ve sample_id kolonu da bulunmuyor."
        )

    metadata_path = find_companion_metadata(
        prediction_path
    )

    if metadata_path is None:
        raise FileNotFoundError(
            f"{region_name}: cache prediction tespit edildi ancak "
            f"companion metadata bulunamadı.\n"
            f"Prediction: {prediction_path}"
        )

    print(
        f"{region_name.upper()} | "
        f"Companion metadata: {metadata_path}"
    )

    metadata = pd.read_csv(
        metadata_path
    )

    if "sample_id" not in metadata.columns:
        raise ValueError(
            f"{region_name}: companion metadata içinde "
            "sample_id kolonu yok."
        )

    source_column = None

    for candidate in SOURCE_KEY_CANDIDATES:

        if candidate not in metadata.columns:
            continue

        test_keys = (
            metadata[candidate]
            .map(canonical_frame_key)
        )

        valid_fraction = (
            test_keys.notna().mean()
        )

        if valid_fraction >= 0.95:
            source_column = candidate
            break

    if source_column is None:
        raise ValueError(
            f"{region_name}: companion metadata içerisinde "
            "upstream frame anahtarı üretilebilecek kolon bulunamadı.\n"
            f"Metadata columns: {list(metadata.columns)}"
        )

    print(
        f"{region_name.upper()} | "
        f"Metadata source column: {source_column}"
    )

    mapping = (
        metadata[
            [
                "sample_id",
                source_column,
            ]
        ]
        .copy()
    )

    mapping["sample_id"] = (
        mapping["sample_id"]
        .astype(str)
        .str.strip()
    )

    mapping["fusion_key"] = (
        mapping[source_column]
        .map(canonical_frame_key)
    )

    mapping = (
        mapping.dropna(
            subset=["fusion_key"]
        )
        .drop_duplicates(
            subset=["sample_id"]
        )
    )

    work = prediction_df.copy()

    work["sample_id"] = (
        work["sample_id"]
        .astype(str)
        .str.strip()
    )

    merged = work.merge(
        mapping[
            [
                "sample_id",
                "fusion_key",
            ]
        ],
        on="sample_id",
        how="left",
        validate="many_to_one",
    )

    missing = int(
        merged["fusion_key"]
        .isna()
        .sum()
    )

    if missing:

        examples = (
            merged.loc[
                merged["fusion_key"].isna(),
                "sample_id",
            ]
            .head(10)
            .tolist()
        )

        raise ValueError(
            f"{region_name}: {missing} prediction sample_id "
            "metadata ile eşleşmedi.\n"
            f"Examples: {examples}"
        )

    return merged["fusion_key"], {
        "key_resolution": "companion_metadata",
        "metadata_path": str(metadata_path),
        "metadata_source_column": str(source_column),
    }


def resolve_fusion_key(
    raw: pd.DataFrame,
    prediction_path: Path,
    region_name: str,
):
    """
    1) Önce prediction CSV içindeki gerçek frame kolonlarını dener.
    2) Eğer path geçici cache ise sample_id -> metadata -> source frame
       zincirini kullanır.
    """

    # --------------------------------------------------------
    # 1) Prediction CSV içinden doğrudan çözmeyi dene
    # --------------------------------------------------------

    direct_candidates = [
        "source_frame",
        "relative_frame_path",
        "frame_path",
        "original_frame",
        "image_path",
        "path",
        "frame_stem",
    ]

    for column in direct_candidates:

        if column not in raw.columns:
            continue

        keys = (
            raw[column]
            .map(canonical_frame_key)
        )

        valid_fraction = float(
            keys.notna().mean()
        )

        # Tam veya neredeyse tam çözülebiliyorsa bunu kullan.
        if valid_fraction >= 0.95:

            print(
                f"{region_name.upper()} | "
                f"Fusion key source: prediction CSV -> {column}"
            )

            return keys, {
                "key_resolution": "prediction_column",
                "key_column": str(column),
            }


    # --------------------------------------------------------
    # 2) Cache/path hash tespit edildi:
    #    sample_id üzerinden metadata'ya dön
    # --------------------------------------------------------

    print(
        f"{region_name.upper()} | "
        "Direct frame key unavailable. "
        "Using sample_id -> companion metadata mapping."
    )

    return build_key_from_metadata(
        prediction_df=raw,
        prediction_path=prediction_path,
        region_name=region_name,
    )


def load_prediction_csv(
    path,
    region_name,
):
    """
    Prediction CSV'yi güvenli biçimde yükler ve upstream frame düzeyine
    normalize eder.

    Kaynak dosyalar SALT OKUNUR:
    - silme yok
    - taşıma yok
    - overwrite yok
    """

    # ========================================================
    # 1) PATH QUALITY GATE
    # ========================================================

    if path is None:
        raise FileNotFoundError(
            f"{region_name}: prediction CSV path is None."
        )

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"{region_name}: prediction CSV does not exist:\n"
            f"{path}"
        )


    # ========================================================
    # 2) READ-ONLY LOAD
    # ========================================================

    raw = pd.read_csv(
        path
    )

    if raw.empty:
        raise ValueError(
            f"{region_name}: prediction CSV is empty:\n"
            f"{path}"
        )


    # ========================================================
    # 3) SEMANTIC COLUMN DISCOVERY
    # ========================================================

    label_col = first_existing(
        raw.columns,
        LABEL_CANDIDATES,
    )

    prob_col = first_existing(
        raw.columns,
        PROB_CANDIDATES,
    )

    if label_col is None:
        raise ValueError(
            f"{region_name}: label column not found.\n"
            f"Columns: {list(raw.columns)}"
        )

    if prob_col is None:
        raise ValueError(
            f"{region_name}: fake probability column not found.\n"
            f"Columns: {list(raw.columns)}"
        )


    # ========================================================
    # 4) NORMALIZE LABELS
    # ========================================================

    normalized_labels = (
        raw[label_col]
        .map(normalize_label)
    )

    invalid_labels = int(
        normalized_labels
        .isna()
        .sum()
    )

    if invalid_labels:
        raise ValueError(
            f"{region_name}: "
            f"{invalid_labels} invalid labels found."
        )


    # ========================================================
    # 5) NORMALIZE PROBABILITIES
    # ========================================================

    probabilities = pd.to_numeric(
        raw[prob_col],
        errors="coerce",
    )

    invalid_probabilities = int(
        probabilities
        .isna()
        .sum()
    )

    if invalid_probabilities:
        raise ValueError(
            f"{region_name}: "
            f"{invalid_probabilities} invalid probabilities found."
        )

    invalid_range = (
        (probabilities < 0)
        | (probabilities > 1)
    )

    if invalid_range.any():
        raise ValueError(
            f"{region_name}: "
            f"{int(invalid_range.sum())} probabilities "
            "outside [0, 1]."
        )


    # ========================================================
    # 6) RESOLVE TRUE UPSTREAM FRAME KEY
    # ========================================================

    fusion_keys, key_audit = (
        resolve_fusion_key(
            raw=raw,
            prediction_path=path,
            region_name=region_name,
        )
    )

    invalid_keys = int(
        fusion_keys
        .isna()
        .sum()
    )

    if invalid_keys:

        raise ValueError(
            f"{region_name}: "
            f"{invalid_keys} fusion keys unresolved."
        )


    # ========================================================
    # 7) BUILD NORMALIZED TABLE
    # ========================================================

    # Audit'te hangi prediction satırından geldiğini gösterebilmek için
    # raw identifier saklanır.
    if "sample_id" in raw.columns:

        raw_identifier = (
            raw["sample_id"]
            .astype(str)
        )

    elif "image_path" in raw.columns:

        raw_identifier = (
            raw["image_path"]
            .astype(str)
        )

    elif "path" in raw.columns:

        raw_identifier = (
            raw["path"]
            .astype(str)
        )

    else:

        raw_identifier = (
            pd.Series(
                np.arange(len(raw)),
                index=raw.index,
            )
            .astype(str)
        )


    normalized = pd.DataFrame(
        {
            "fusion_key": fusion_keys,
            f"label_{region_name}": (
                normalized_labels
                .astype(int)
            ),
            f"p_{region_name}": (
                probabilities
                .astype(float)
            ),
            f"raw_key_{region_name}": (
                raw_identifier
            ),
        }
    )


    # ========================================================
    # 8) FRAME-LEVEL AGGREGATION
    # ========================================================

    grouped = (
        normalized
        .groupby(
            "fusion_key",
            as_index=False,
        )
        .agg(
            **{
                f"label_min_{region_name}": (
                    f"label_{region_name}",
                    "min",
                ),

                f"label_max_{region_name}": (
                    f"label_{region_name}",
                    "max",
                ),

                f"p_{region_name}": (
                    f"p_{region_name}",
                    FRAME_AGGREGATION,
                ),

                f"raw_key_{region_name}": (
                    f"raw_key_{region_name}",
                    "first",
                ),

                f"roi_count_{region_name}": (
                    f"p_{region_name}",
                    "size",
                ),
            }
        )
    )


    # ========================================================
    # 9) LABEL CONSISTENCY QUALITY GATE
    # ========================================================

    inconsistent_labels = (
        grouped[
            f"label_min_{region_name}"
        ]
        != grouped[
            f"label_max_{region_name}"
        ]
    )

    if inconsistent_labels.any():

        raise ValueError(
            f"{region_name}: "
            f"{int(inconsistent_labels.sum())} upstream frames "
            "contain conflicting labels."
        )


    grouped[
        f"label_{region_name}"
    ] = (
        grouped[
            f"label_min_{region_name}"
        ]
        .astype(int)
    )


    grouped = grouped.drop(
        columns=[
            f"label_min_{region_name}",
            f"label_max_{region_name}",
        ]
    )


    # ========================================================
    # 10) AUDIT
    # ========================================================

    audit = {
        "region": region_name,

        "prediction_path": str(
            path
        ),

        "source_rows": int(
            len(raw)
        ),

        "unique_upstream_frames": int(
            len(grouped)
        ),

        "multi_roi_frames": int(
            (
                grouped[
                    f"roi_count_{region_name}"
                ]
                > 1
            )
            .sum()
        ),

        "max_roi_count_per_frame": int(
            grouped[
                f"roi_count_{region_name}"
            ]
            .max()
        ),

        "label_column": str(
            label_col
        ),

        "probability_column": str(
            prob_col
        ),

        "frame_aggregation": (
            FRAME_AGGREGATION
        ),

        **key_audit,
    }


    # ========================================================
    # 11) HUMAN-READABLE AUDIT
    # ========================================================

    print(
        f"{region_name.upper():5s} | "
        f"Rows={len(raw)} | "
        f"Frames={len(grouped)} | "
        f"Label={label_col} | "
        f"Probability={prob_col} | "
        f"Key={key_audit['key_resolution']}"
    )


    return grouped, audit


def align_three(eye_path, brow_path, mouth_path):
    eye, eye_audit = load_prediction_csv(eye_path, "eye")
    brow, brow_audit = load_prediction_csv(brow_path, "brow")
    mouth, mouth_audit = load_prediction_csv(mouth_path, "mouth")

    aligned = eye.merge(
        brow,
        on="fusion_key",
        how="inner",
        validate="one_to_one",
    )
    aligned = aligned.merge(
        mouth,
        on="fusion_key",
        how="inner",
        validate="one_to_one",
    )

    if aligned.empty:
        raise RuntimeError(
            "Eye/Brow/Mouth upstream frame intersection is empty."
        )

    label_columns = [
        "label_eye",
        "label_brow",
        "label_mouth",
    ]

    label_matrix = aligned[label_columns].astype(int)
    consistent_labels = label_matrix.nunique(axis=1).eq(1)

    if not consistent_labels.all():
        bad = aligned.loc[
            ~consistent_labels,
            ["fusion_key"] + label_columns,
        ].head(10)

        raise ValueError(
            f"{int((~consistent_labels).sum())} aligned frames have "
            f"inconsistent region labels.\n{bad}"
        )

    aligned["label"] = label_matrix.iloc[:, 0].astype(int)

    required_probabilities = [
        "p_eye",
        "p_brow",
        "p_mouth",
    ]

    for column in required_probabilities:
        values = pd.to_numeric(
            aligned[column],
            errors="coerce",
        )

        if values.isna().any():
            raise ValueError(
                f"{column}: NaN probability found after alignment."
            )

        if ((values < 0) | (values > 1)).any():
            raise ValueError(
                f"{column}: probability outside [0, 1] after alignment."
            )

        aligned[column] = values.astype(float)

    aligned = (
        aligned.sort_values("fusion_key")
        .reset_index(drop=True)
    )

    counts = {
        "eye_frames": int(len(eye)),
        "brow_frames": int(len(brow)),
        "mouth_frames": int(len(mouth)),
        "common_frames": int(len(aligned)),
        "real_common_frames": int((aligned["label"] == 0).sum()),
        "fake_common_frames": int((aligned["label"] == 1).sum()),
    }

    if counts["real_common_frames"] == 0 or counts["fake_common_frames"] == 0:
        raise RuntimeError(
            "The aligned common set must contain both REAL and FAKE classes."
        )

    audit = {
        "counts": counts,
        "eye": eye_audit,
        "brow": brow_audit,
        "mouth": mouth_audit,
    }

    return aligned, audit


def compute_metrics(y_true, probabilities, threshold=DECISION_THRESHOLD):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)

    if y_true.shape[0] != probabilities.shape[0]:
        raise ValueError(
            "y_true and probabilities have different lengths."
        )

    if not np.isfinite(probabilities).all():
        raise ValueError(
            "NaN/Inf found in probabilities."
        )

    predicted = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predicted,
        labels=[0, 1],
    ).ravel()

    return {
        "n": int(len(y_true)),
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, predicted)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, predicted)
        ),
        "precision": float(
            precision_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "specificity": float(
            tn / (tn + fp)
            if (tn + fp)
            else np.nan
        ),
        "f1": float(
            f1_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [11]:
# ============================================================
# 5) PRE-FLIGHT QUALITY GATES
# ============================================================

EXPECTED_KEY_CASES = {
    "fake_test_00000__face_00.jpg": "fake_test_00000",
    "fake_test_00000.jpg": "fake_test_00000",
    "fake_test_fake_test_00000_face00.png": "fake_test_00000",
    "real_test_00125__face_00.jpg": "real_test_00125",
    "real_test_00125.jpg": "real_test_00125",
    "real_test_real_test_00125_face00.png": "real_test_00125",
}

for raw_key, expected_key in EXPECTED_KEY_CASES.items():
    actual_key = canonical_frame_key(raw_key)
    assert actual_key == expected_key, (
        f"Frame-key test failed: {raw_key} -> {actual_key}; "
        f"expected {expected_key}"
    )

print("Frame-key normalization test: PASSED")


path_audit_rows = []

for family, cfg in MODEL_FAMILIES.items():
    for region in ("eye", "brow", "mouth"):
        path = cfg[f"{region}_test"]
        exists = path is not None and Path(path).is_file()

        path_audit_rows.append(
            {
                "model_family": family,
                "region": region,
                "path": str(path),
                "exists": bool(exists),
            }
        )

path_audit = pd.DataFrame(path_audit_rows)

if not path_audit["exists"].all():
    missing = path_audit.loc[
        ~path_audit["exists"],
        ["model_family", "region", "path"],
    ]
    raise FileNotFoundError(
        "One or more configured test prediction files are missing:\n"
        + missing.to_string(index=False)
    )

print("Configured test prediction paths: PASSED")
display(path_audit)

atomic_write_csv(
    path_audit,
    RUN_DIR / "audit" / "configured_paths.csv",
)

Frame-key normalization test: PASSED
Configured test prediction paths: PASSED


,model_family,region,path,exists
0,swinv2_tiny,eye,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
1,swinv2_tiny,brow,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
2,swinv2_tiny,mouth,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
3,efficientnet_b0,eye,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
4,efficientnet_b0,brow,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
5,efficientnet_b0,mouth,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
6,swinv2_texture,eye,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
7,swinv2_texture,brow,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
8,swinv2_texture,mouth,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True


In [12]:
# ============================================================
# 6) VISUALIZATION HELPERS
# ============================================================

DISPLAY_NAMES = {
    "eye_only": "Eye only",
    "brow_only": "Brow only",
    "mouth_only": "Mouth only",
    "fusion": "Equal Soft Voting",
}

PROBABILITY_COLUMNS = {
    "eye_only": "p_eye",
    "brow_only": "p_brow",
    "mouth_only": "p_mouth",
    "fusion": "fusion_probability",
}


def plot_coverage(audit, family_name, figure_dir):
    counts = audit["counts"]

    labels = [
        "Eye",
        "Brow",
        "Mouth",
        "Common",
    ]
    values = [
        counts["eye_frames"],
        counts["brow_frames"],
        counts["mouth_frames"],
        counts["common_frames"],
    ]

    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.bar(labels, values)

    ax.set_title(
        f"{family_name} — Frame Coverage",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax.set_ylabel("Number of Upstream Frames", fontsize=11)
    ax.grid(axis="y", alpha=0.25)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(value),
            ha="center",
            va="bottom",
            fontsize=11,
        )

    fig.tight_layout()
    save_figure(fig, figure_dir / "frame_coverage")


def plot_class_distribution(df, family_name, figure_dir):
    counts = (
        df["label"]
        .value_counts()
        .reindex([0, 1], fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(
        ["REAL", "FAKE"],
        counts.values,
    )

    ax.set_title(
        f"{family_name} — Common-Set Class Distribution",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax.set_ylabel("Number of Frames", fontsize=11)
    ax.grid(axis="y", alpha=0.25)

    for bar, value in zip(bars, counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(int(value)),
            ha="center",
            va="bottom",
            fontsize=11,
        )

    fig.tight_layout()
    save_figure(
        fig,
        figure_dir / "common_set_class_distribution",
    )


def plot_roc_comparison(df, family_name, figure_dir):
    y_true = df["label"].to_numpy()

    fig, ax = plt.subplots(figsize=(9, 7))

    for evaluation, column in PROBABILITY_COLUMNS.items():
        probabilities = df[column].to_numpy()
        fpr, tpr, _ = roc_curve(y_true, probabilities)
        auc_value = roc_auc_score(
            y_true,
            probabilities,
        )

        ax.plot(
            fpr,
            tpr,
            linewidth=2,
            label=(
                f"{DISPLAY_NAMES[evaluation]} "
                f"(AUC={auc_value:.3f})"
            ),
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Chance",
    )
    ax.set_title(
        f"{family_name} — ROC Comparison",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax.set_xlabel("False Positive Rate", fontsize=11)
    ax.set_ylabel("True Positive Rate", fontsize=11)
    ax.legend(frameon=True)
    ax.grid(alpha=0.25)

    fig.tight_layout()
    save_figure(fig, figure_dir / "roc_comparison")


def plot_pr_comparison(df, family_name, figure_dir):
    y_true = df["label"].to_numpy()

    fig, ax = plt.subplots(figsize=(9, 7))

    for evaluation, column in PROBABILITY_COLUMNS.items():
        probabilities = df[column].to_numpy()
        precision, recall, _ = precision_recall_curve(
            y_true,
            probabilities,
        )
        ap_value = average_precision_score(
            y_true,
            probabilities,
        )

        ax.plot(
            recall,
            precision,
            linewidth=2,
            label=(
                f"{DISPLAY_NAMES[evaluation]} "
                f"(AP={ap_value:.3f})"
            ),
        )

    ax.set_title(
        f"{family_name} — Precision–Recall Comparison",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax.set_xlabel("Recall", fontsize=11)
    ax.set_ylabel("Precision", fontsize=11)
    ax.legend(frameon=True)
    ax.grid(alpha=0.25)

    fig.tight_layout()
    save_figure(fig, figure_dir / "precision_recall_comparison")


def plot_fusion_confusion_matrix(df, family_name, figure_dir):
    y_true = df["label"].to_numpy()
    predicted = (
        df["fusion_probability"].to_numpy()
        >= DECISION_THRESHOLD
    ).astype(int)

    matrix = confusion_matrix(
        y_true,
        predicted,
        labels=[0, 1],
    )

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix)

    ax.set_xticks([0, 1], labels=["REAL", "FAKE"])
    ax.set_yticks([0, 1], labels=["REAL", "FAKE"])
    ax.set_xlabel("Predicted Class", fontsize=11)
    ax.set_ylabel("True Class", fontsize=11)
    ax.set_title(
        f"{family_name} — Equal Soft Voting Confusion Matrix",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    for row in range(2):
        for column in range(2):
            ax.text(
                column,
                row,
                str(int(matrix[row, column])),
                ha="center",
                va="center",
                fontsize=12,
            )

    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    save_figure(
        fig,
        figure_dir / "fusion_confusion_matrix",
    )


def plot_probability_histograms(df, family_name, figure_dir):
    fig, ax = plt.subplots(figsize=(10, 7))

    for evaluation, column in PROBABILITY_COLUMNS.items():
        ax.hist(
            df[column].to_numpy(),
            bins=20,
            alpha=0.35,
            label=DISPLAY_NAMES[evaluation],
        )

    ax.axvline(
        DECISION_THRESHOLD,
        linestyle="--",
        linewidth=2,
        label=f"Decision Threshold ({DECISION_THRESHOLD:.2f})",
    )

    ax.set_title(
        f"{family_name} — Probability Distributions",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax.set_xlabel("Predicted FAKE Probability", fontsize=11)
    ax.set_ylabel("Frame Count", fontsize=11)
    ax.legend(frameon=True)
    ax.grid(alpha=0.25)

    fig.tight_layout()
    save_figure(
        fig,
        figure_dir / "probability_distributions",
    )


def plot_probability_by_class(df, family_name, figure_dir):
    real = df.loc[
        df["label"] == 0,
        "fusion_probability",
    ].to_numpy()
    fake = df.loc[
        df["label"] == 1,
        "fusion_probability",
    ].to_numpy()

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.boxplot(
        [real, fake],
        tick_labels=["REAL", "FAKE"],
        showmeans=True,
    )

    ax.axhline(
        DECISION_THRESHOLD,
        linestyle="--",
        linewidth=2,
        label=f"Decision Threshold ({DECISION_THRESHOLD:.2f})",
    )
    ax.set_title(
        f"{family_name} — Fusion Probability by True Class",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax.set_ylabel("Predicted FAKE Probability", fontsize=11)
    ax.legend(frameon=True)
    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()
    save_figure(
        fig,
        figure_dir / "fusion_probability_by_true_class",
    )


def plot_region_correlation(df, family_name, figure_dir):
    columns = [
        "p_eye",
        "p_brow",
        "p_mouth",
        "fusion_probability",
    ]

    labels = [
        "Eye",
        "Brow",
        "Mouth",
        "Fusion",
    ]

    correlation = df[columns].corr()

    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(
        correlation.to_numpy(),
        vmin=-1,
        vmax=1,
    )

    ax.set_xticks(
        range(len(labels)),
        labels=labels,
        rotation=30,
        ha="right",
    )
    ax.set_yticks(
        range(len(labels)),
        labels=labels,
    )

    ax.set_title(
        f"{family_name} — Prediction Correlation",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    for row in range(len(labels)):
        for column in range(len(labels)):
            ax.text(
                column,
                row,
                f"{correlation.iloc[row, column]:.2f}",
                ha="center",
                va="center",
                fontsize=11,
            )

    fig.colorbar(
        image,
        ax=ax,
        label="Pearson Correlation",
    )

    fig.tight_layout()
    save_figure(
        fig,
        figure_dir / "prediction_correlation",
    )


def plot_region_disagreement(df, family_name, figure_dir):
    votes = pd.DataFrame(
        {
            "Eye": (
                df["p_eye"] >= DECISION_THRESHOLD
            ).astype(int),
            "Brow": (
                df["p_brow"] >= DECISION_THRESHOLD
            ).astype(int),
            "Mouth": (
                df["p_mouth"] >= DECISION_THRESHOLD
            ).astype(int),
        }
    )

    pattern = (
        votes.astype(str)
        .agg("-".join, axis=1)
        .map(
            {
                "0-0-0": "All REAL",
                "1-1-1": "All FAKE",
                "0-0-1": "Mouth only FAKE",
                "0-1-0": "Brow only FAKE",
                "1-0-0": "Eye only FAKE",
                "0-1-1": "Brow + Mouth FAKE",
                "1-0-1": "Eye + Mouth FAKE",
                "1-1-0": "Eye + Brow FAKE",
            }
        )
    )

    counts = pattern.value_counts()

    fig, ax = plt.subplots(figsize=(11, 7))
    bars = ax.barh(
        counts.index,
        counts.values,
    )

    ax.set_title(
        f"{family_name} — Regional Decision Agreement",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax.set_xlabel("Number of Common Frames", fontsize=11)
    ax.grid(axis="x", alpha=0.25)

    for bar, value in zip(bars, counts.values):
        ax.text(
            bar.get_width(),
            bar.get_y() + bar.get_height() / 2,
            str(int(value)),
            va="center",
            ha="left",
            fontsize=10,
        )

    fig.tight_layout()
    save_figure(
        fig,
        figure_dir / "regional_decision_agreement",
    )


def plot_metric_comparison(metrics_df, family_name, figure_dir):
    selected = metrics_df[
        [
            "evaluation",
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "f1",
        ]
    ].copy()

    selected["evaluation"] = selected["evaluation"].map(
        DISPLAY_NAMES
    )

    plot_df = selected.set_index("evaluation")

    fig, ax = plt.subplots(figsize=(11, 7))
    plot_df.plot(
        kind="bar",
        ax=ax,
    )

    ax.set_title(
        f"{family_name} — Regional vs Fusion Metrics",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Score", fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=20)
    ax.legend(
        [
            "ROC-AUC",
            "PR-AUC",
            "Balanced Accuracy",
            "F1",
        ],
        frameon=True,
    )
    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()
    save_figure(
        fig,
        figure_dir / "regional_vs_fusion_metrics",
    )


def create_family_figures(
    aligned_df,
    metrics_df,
    audit,
    family_name,
    figure_dir,
):
    figure_dir.mkdir(parents=True, exist_ok=True)

    plot_coverage(
        audit,
        family_name,
        figure_dir,
    )
    plot_class_distribution(
        aligned_df,
        family_name,
        figure_dir,
    )
    plot_roc_comparison(
        aligned_df,
        family_name,
        figure_dir,
    )
    plot_pr_comparison(
        aligned_df,
        family_name,
        figure_dir,
    )
    plot_fusion_confusion_matrix(
        aligned_df,
        family_name,
        figure_dir,
    )
    plot_probability_histograms(
        aligned_df,
        family_name,
        figure_dir,
    )
    plot_probability_by_class(
        aligned_df,
        family_name,
        figure_dir,
    )
    plot_region_correlation(
        aligned_df,
        family_name,
        figure_dir,
    )
    plot_region_disagreement(
        aligned_df,
        family_name,
        figure_dir,
    )
    plot_metric_comparison(
        metrics_df,
        family_name,
        figure_dir,
    )

In [13]:
# ============================================================
# 7) EQUAL SOFT VOTING — ALL THREE MODEL FAMILIES
# ============================================================

all_metric_rows = []
family_audit_rows = []

for family, cfg in MODEL_FAMILIES.items():
    print("\n" + "=" * 90)
    print(f"MODEL FAMILY: {family}")
    print("=" * 90)

    aligned, audit = align_three(
        cfg["eye_test"],
        cfg["brow_test"],
        cfg["mouth_test"],
    )

    aligned["fusion_probability"] = (
        aligned["p_eye"]
        + aligned["p_brow"]
        + aligned["p_mouth"]
    ) / 3.0

    if not np.isfinite(
        aligned["fusion_probability"].to_numpy()
    ).all():
        raise RuntimeError(
            f"{family}: NaN/Inf found in fusion probabilities."
        )

    if (
        (aligned["fusion_probability"] < 0)
        | (aligned["fusion_probability"] > 1)
    ).any():
        raise RuntimeError(
            f"{family}: fusion probability outside [0, 1]."
        )

    family_dir = RUN_DIR / family
    figure_dir = family_dir / "figures"
    metric_rows = []

    for evaluation, probability_column in PROBABILITY_COLUMNS.items():
        metrics = compute_metrics(
            aligned["label"],
            aligned[probability_column],
            threshold=DECISION_THRESHOLD,
        )

        row = {
            "model_family": family,
            "method": METHOD_NAME,
            "evaluation": evaluation,
            "threshold_source": "fixed_predefined_0.50",
            **metrics,
        }

        metric_rows.append(row)
        all_metric_rows.append(row)

    family_metrics = pd.DataFrame(metric_rows)

    # Audit before saving results.
    common_count = audit["counts"]["common_frames"]

    if len(aligned) != common_count:
        raise RuntimeError(
            f"{family}: alignment accounting mismatch."
        )

    if aligned["fusion_key"].duplicated().any():
        raise RuntimeError(
            f"{family}: duplicate fusion_key found after alignment."
        )

    if not set(aligned["label"].unique()).issubset({0, 1}):
        raise RuntimeError(
            f"{family}: invalid final labels found."
        )

    atomic_write_csv(
        aligned,
        family_dir
        / "predictions"
        / "aligned_test_predictions.csv",
    )

    atomic_write_csv(
        family_metrics,
        family_dir
        / "metrics"
        / "test_metrics.csv",
    )

    atomic_write_json(
        {
            "run_id": RUN_ID,
            "method": METHOD_NAME,
            "model_family": family,
            "decision_threshold": DECISION_THRESHOLD,
            "threshold_source": "fixed_predefined_0.50",
            "frame_aggregation": FRAME_AGGREGATION,
            "alignment_audit": audit,
            "fusion_metrics": family_metrics.loc[
                family_metrics["evaluation"] == "fusion"
            ].iloc[0].to_dict(),
        },
        family_dir
        / "audit"
        / "run_audit.json",
    )

    create_family_figures(
        aligned_df=aligned,
        metrics_df=family_metrics,
        audit=audit,
        family_name=family,
        figure_dir=figure_dir,
    )

    family_audit_rows.append(
        {
            "model_family": family,
            **audit["counts"],
            "eye_multi_roi_frames": audit["eye"]["multi_roi_frames"],
            "brow_multi_roi_frames": audit["brow"]["multi_roi_frames"],
            "mouth_multi_roi_frames": audit["mouth"]["multi_roi_frames"],
        }
    )

    print(
        f"Common frames: {audit['counts']['common_frames']} "
        f"(REAL={audit['counts']['real_common_frames']}, "
        f"FAKE={audit['counts']['fake_common_frames']})"
    )

    display(
        family_metrics[
            [
                "evaluation",
                "n",
                "accuracy",
                "balanced_accuracy",
                "precision",
                "recall",
                "specificity",
                "f1",
                "roc_auc",
                "pr_auc",
            ]
        ]
    )


all_metrics = pd.DataFrame(all_metric_rows)
family_audit = pd.DataFrame(family_audit_rows)

atomic_write_csv(
    all_metrics,
    RUN_DIR / "metrics" / "all_model_families_metrics.csv",
)

atomic_write_csv(
    family_audit,
    RUN_DIR / "audit" / "family_alignment_summary.csv",
)

print("\nAll three model families completed.")


MODEL FAMILY: swinv2_tiny
EYE | Fusion key source: prediction CSV -> image_path
EYE   | Rows=302 | Frames=292 | Label=true_label | Probability=fake_probability | Key=prediction_column
BROW | Fusion key source: prediction CSV -> image_path
BROW  | Rows=196 | Frames=196 | Label=true_label | Probability=fake_probability | Key=prediction_column
MOUTH | Fusion key source: prediction CSV -> image_path
MOUTH | Rows=302 | Frames=292 | Label=true_label | Probability=fake_probability | Key=prediction_column
Common frames: 196 (REAL=101, FAKE=95)


,evaluation,n,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc
0,eye_only,196,0.744898,0.743408,0.758621,0.694737,0.792079,0.725275,0.782908,0.786848
1,brow_only,196,0.668367,0.667587,0.663043,0.642105,0.693069,0.652406,0.681501,0.654312
2,mouth_only,196,0.806122,0.804065,0.843373,0.736842,0.871287,0.786517,0.880250,0.870051
3,fusion,196,0.770408,0.768161,0.804878,0.694737,0.841584,0.745763,0.858676,0.865218



MODEL FAMILY: efficientnet_b0
EYE | Direct frame key unavailable. Using sample_id -> companion metadata mapping.
EYE | Companion metadata: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/20260808_0803_eye_efficientnet_b0_seed42/artifacts/eligible_metadata.csv
EYE | Metadata source column: source_frame
EYE   | Rows=302 | Frames=292 | Label=label | Probability=prob_fake | Key=companion_metadata
BROW | Direct frame key unavailable. Using sample_id -> companion metadata mapping.
BROW | Companion metadata: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/Sonuçlar/20260808_1248_eyebrow_efficientnet_b0_seed42/artifacts/eligible_metadata.csv
BROW | Metadata source column: input_path
BROW  | Rows=196 | Frames=196 | Label=label | Probability=prob_fake | Key=companion_metadata
MOUTH | Fusion key source: prediction CSV -> path
MOUTH | Rows=302 | Frames=292 | Label=label | Probability=prob_fake | Key=prediction_column
Common frames: 196 (REA

,evaluation,n,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc
0,eye_only,196,0.673469,0.673476,0.659794,0.673684,0.673267,0.666667,0.731318,0.728882
1,brow_only,196,0.551020,0.552788,0.532110,0.610526,0.495050,0.568627,0.587389,0.576110
2,mouth_only,196,0.755102,0.754560,0.752688,0.736842,0.772277,0.744681,0.833351,0.837753
3,fusion,196,0.724490,0.723919,0.720430,0.705263,0.742574,0.712766,0.801668,0.805531



MODEL FAMILY: swinv2_texture
EYE | Fusion key source: prediction CSV -> image_path
EYE   | Rows=302 | Frames=292 | Label=target | Probability=probability_fake | Key=prediction_column
BROW | Fusion key source: prediction CSV -> image_path
BROW  | Rows=196 | Frames=196 | Label=target | Probability=probability_fake | Key=prediction_column
MOUTH | Fusion key source: prediction CSV -> image_path
MOUTH | Rows=302 | Frames=292 | Label=target | Probability=probability_fake | Key=prediction_column
Common frames: 196 (REAL=101, FAKE=95)


,evaluation,n,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc
0,eye_only,196,0.642857,0.640021,0.658228,0.547368,0.732673,0.597701,0.675560,0.644614
1,brow_only,196,0.622449,0.619281,0.636364,0.515789,0.722772,0.569767,0.668786,0.670791
2,mouth_only,196,0.724490,0.723293,0.730337,0.684211,0.762376,0.706522,0.789682,0.751422
3,fusion,196,0.693878,0.692027,0.705882,0.631579,0.752475,0.666667,0.752267,0.736103



All three model families completed.


In [14]:
# ============================================================
# 8) CROSS-FAMILY VISUAL COMPARISONS
# ============================================================

overall_figure_dir = RUN_DIR / "figures"
overall_figure_dir.mkdir(parents=True, exist_ok=True)

fusion_only = (
    all_metrics.loc[
        all_metrics["evaluation"] == "fusion"
    ]
    .copy()
    .sort_values("model_family")
)

# ------------------------------------------------------------
# A) Fusion metric comparison across model families
# ------------------------------------------------------------

metric_columns = [
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "f1",
]

plot_data = (
    fusion_only[
        ["model_family"] + metric_columns
    ]
    .set_index("model_family")
)

fig, ax = plt.subplots(figsize=(12, 7))
plot_data.plot(
    kind="bar",
    ax=ax,
)

ax.set_title(
    "Equal Soft Voting — Model Family Comparison",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax.set_xlabel("Model Family", fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=15)
ax.legend(
    [
        "ROC-AUC",
        "PR-AUC",
        "Balanced Accuracy",
        "F1",
    ],
    frameon=True,
)
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
save_figure(
    fig,
    overall_figure_dir
    / "equal_soft_voting_model_family_comparison",
)


# ------------------------------------------------------------
# B) Common frame counts across model families
# ------------------------------------------------------------

coverage_data = (
    family_audit[
        [
            "model_family",
            "eye_frames",
            "brow_frames",
            "mouth_frames",
            "common_frames",
        ]
    ]
    .set_index("model_family")
)

fig, ax = plt.subplots(figsize=(12, 7))
coverage_data.plot(
    kind="bar",
    ax=ax,
)

ax.set_title(
    "Model Family — Available and Common Frame Counts",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax.set_xlabel("Model Family", fontsize=11)
ax.set_ylabel("Number of Upstream Frames", fontsize=11)
ax.tick_params(axis="x", rotation=15)
ax.legend(
    [
        "Eye",
        "Brow",
        "Mouth",
        "Common",
    ],
    frameon=True,
)
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
save_figure(
    fig,
    overall_figure_dir
    / "model_family_frame_coverage_comparison",
)


# ------------------------------------------------------------
# C) Fusion improvement over best single region
# ------------------------------------------------------------

improvement_rows = []

for family in all_metrics["model_family"].unique():
    subset = all_metrics[
        all_metrics["model_family"] == family
    ]

    single = subset[
        subset["evaluation"].isin(
            [
                "eye_only",
                "brow_only",
                "mouth_only",
            ]
        )
    ]

    fusion = subset[
        subset["evaluation"] == "fusion"
    ].iloc[0]

    best_single_auc = single["roc_auc"].max()
    best_single_f1 = single["f1"].max()

    improvement_rows.append(
        {
            "model_family": family,
            "roc_auc_gain_vs_best_single": (
                fusion["roc_auc"]
                - best_single_auc
            ),
            "f1_gain_vs_best_single": (
                fusion["f1"]
                - best_single_f1
            ),
        }
    )

improvement_df = pd.DataFrame(improvement_rows)

atomic_write_csv(
    improvement_df,
    RUN_DIR
    / "metrics"
    / "fusion_gain_vs_best_single_region.csv",
)

fig, ax = plt.subplots(figsize=(11, 7))

improvement_df.set_index(
    "model_family"
).plot(
    kind="bar",
    ax=ax,
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
)
ax.set_title(
    "Equal Soft Voting — Gain vs Best Single Region",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax.set_xlabel("Model Family", fontsize=11)
ax.set_ylabel("Absolute Metric Difference", fontsize=11)
ax.tick_params(axis="x", rotation=15)
ax.legend(
    [
        "ROC-AUC Gain",
        "F1 Gain",
    ],
    frameon=True,
)
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
save_figure(
    fig,
    overall_figure_dir
    / "fusion_gain_vs_best_single_region",
)

display(fusion_only)
display(improvement_df)

,model_family,method,evaluation,threshold_source,n,threshold,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc,tn,fp,fn,tp
7,efficientnet_b0,01_equal_soft_voting,fusion,fixed_predefined_0.50,196,0.5,0.724490,0.723919,0.720430,0.705263,0.742574,0.712766,0.801668,0.805531,75,26,28,67
11,swinv2_texture,01_equal_soft_voting,fusion,fixed_predefined_0.50,196,0.5,0.693878,0.692027,0.705882,0.631579,0.752475,0.666667,0.752267,0.736103,76,25,35,60
3,swinv2_tiny,01_equal_soft_voting,fusion,fixed_predefined_0.50,196,0.5,0.770408,0.768161,0.804878,0.694737,0.841584,0.745763,0.858676,0.865218,85,16,29,66


,model_family,roc_auc_gain_vs_best_single,f1_gain_vs_best_single
0,swinv2_tiny,-0.021574,-0.040754
1,efficientnet_b0,-0.031683,-0.031915
2,swinv2_texture,-0.037415,-0.039855


In [15]:
# ============================================================
# 9) FINAL QUALITY GATES + OUTPUT MANIFEST
# ============================================================

required_metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "pr_auc",
]

missing_metric_columns = [
    column
    for column in required_metric_columns
    if column not in all_metrics.columns
]

if missing_metric_columns:
    raise RuntimeError(
        f"Missing metric columns: {missing_metric_columns}"
    )

numeric_metrics = all_metrics[
    required_metric_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

if not np.isfinite(
    numeric_metrics.to_numpy()
).all():
    raise RuntimeError(
        "NaN/Inf found in final metric table."
    )

bounded_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "pr_auc",
]

for column in bounded_metrics:
    invalid = (
        (all_metrics[column] < 0)
        | (all_metrics[column] > 1)
    )

    if invalid.any():
        raise RuntimeError(
            f"{column}: value outside [0, 1]."
        )


# Verify all model families produced all four evaluations.
expected_evaluations = {
    "eye_only",
    "brow_only",
    "mouth_only",
    "fusion",
}

for family in MODEL_FAMILIES:
    actual = set(
        all_metrics.loc[
            all_metrics["model_family"] == family,
            "evaluation",
        ]
    )

    if actual != expected_evaluations:
        raise RuntimeError(
            f"{family}: expected evaluations "
            f"{expected_evaluations}, found {actual}"
        )


# Verify figure resolution and produce a manifest.
manifest_rows = []

for path in sorted(RUN_DIR.rglob("*")):
    if not path.is_file():
        continue

    record = {
        "relative_path": str(
            path.relative_to(RUN_DIR)
        ),
        "size_bytes": int(path.stat().st_size),
        "suffix": path.suffix.lower(),
    }

    if path.suffix.lower() == ".png":
        with Image.open(path) as image:
            record["width_px"] = int(image.width)
            record["height_px"] = int(image.height)

            if min(image.size) < MIN_FIGURE_SHORT_EDGE_PX:
                raise RuntimeError(
                    f"Figure below minimum resolution: "
                    f"{path} -> {image.size}"
                )

    manifest_rows.append(record)

manifest = pd.DataFrame(manifest_rows)

atomic_write_csv(
    manifest,
    RUN_DIR / "output_manifest.csv",
)

final_summary = {
    "run_id": RUN_ID,
    "method": METHOD_NAME,
    "seed": SEED,
    "decision_threshold": DECISION_THRESHOLD,
    "threshold_source": "fixed_predefined_0.50",
    "frame_aggregation": FRAME_AGGREGATION,
    "model_families_completed": sorted(
        all_metrics["model_family"].unique().tolist()
    ),
    "total_metric_rows": int(len(all_metrics)),
    "total_output_files_before_manifest": int(len(manifest_rows)),
    "quality_gates": "PASSED",
}

atomic_write_json(
    final_summary,
    RUN_DIR / "run_summary.json",
)

print("=" * 90)
print("FINAL QUALITY GATES: PASSED")
print("=" * 90)
print(f"Run ID : {RUN_ID}")
print(f"Output : {RUN_DIR}")
print(
    f"PNG figures: "
    f"{int((manifest['suffix'] == '.png').sum())}"
)
print(
    f"SVG figures: "
    f"{int((manifest['suffix'] == '.svg').sum())}"
)

display(
    all_metrics.sort_values(
        ["model_family", "evaluation"]
    )
)

FINAL QUALITY GATES: PASSED
Run ID : 20260809_162333_region_equal_soft_voting_seed42
Output : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/01_equal_soft_voting/20260809_162333_region_equal_soft_voting_seed42
PNG figures: 33
SVG figures: 33


,model_family,method,evaluation,threshold_source,n,threshold,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc,tn,fp,fn,tp
5,efficientnet_b0,01_equal_soft_voting,brow_only,fixed_predefined_0.50,196,0.5,0.551020,0.552788,0.532110,0.610526,0.495050,0.568627,0.587389,0.576110,50,51,37,58
4,efficientnet_b0,01_equal_soft_voting,eye_only,fixed_predefined_0.50,196,0.5,0.673469,0.673476,0.659794,0.673684,0.673267,0.666667,0.731318,0.728882,68,33,31,64
7,efficientnet_b0,01_equal_soft_voting,fusion,fixed_predefined_0.50,196,0.5,0.724490,0.723919,0.720430,0.705263,0.742574,0.712766,0.801668,0.805531,75,26,28,67
6,efficientnet_b0,01_equal_soft_voting,mouth_only,fixed_predefined_0.50,196,0.5,0.755102,0.754560,0.752688,0.736842,0.772277,0.744681,0.833351,0.837753,78,23,25,70
9,swinv2_texture,01_equal_soft_voting,brow_only,fixed_predefined_0.50,196,0.5,0.622449,0.619281,0.636364,0.515789,0.722772,0.569767,0.668786,0.670791,73,28,46,49
8,swinv2_texture,01_equal_soft_voting,eye_only,fixed_predefined_0.50,196,0.5,0.642857,0.640021,0.658228,0.547368,0.732673,0.597701,0.675560,0.644614,74,27,43,52
11,swinv2_texture,01_equal_soft_voting,fusion,fixed_predefined_0.50,196,0.5,0.693878,0.692027,0.705882,0.631579,0.752475,0.666667,0.752267,0.736103,76,25,35,60
10,swinv2_texture,01_equal_soft_voting,mouth_only,fixed_predefined_0.50,196,0.5,0.724490,0.723293,0.730337,0.684211,0.762376,0.706522,0.789682,0.751422,77,24,30,65
1,swinv2_tiny,01_equal_soft_voting,brow_only,fixed_predefined_0.50,196,0.5,0.668367,0.667587,0.663043,0.642105,0.693069,0.652406,0.681501,0.654312,70,31,34,61
0,swinv2_tiny,01_equal_soft_voting,eye_only,fixed_predefined_0.50,196,0.5,0.744898,0.743408,0.758621,0.694737,0.792079,0.725275,0.782908,0.786848,80,21,29,66


## Output structure

Her çalıştırmada yeni bir `RUN_ID` klasörü oluşur; böylece önceki fusion çıktıları
üzerine yazılmaz.

```text
Fusion_Experiments/
└── 01_equal_soft_voting/
    └── <RUN_ID>/
        ├── environment.json
        ├── run_summary.json
        ├── output_manifest.csv
        ├── audit/
        │   ├── configured_paths.csv
        │   └── family_alignment_summary.csv
        ├── metrics/
        │   ├── all_model_families_metrics.csv
        │   └── fusion_gain_vs_best_single_region.csv
        ├── figures/
        │   ├── equal_soft_voting_model_family_comparison.*
        │   ├── model_family_frame_coverage_comparison.*
        │   └── fusion_gain_vs_best_single_region.*
        └── <model_family>/
            ├── audit/run_audit.json
            ├── predictions/aligned_test_predictions.csv
            ├── metrics/test_metrics.csv
            └── figures/
                ├── frame_coverage.*
                ├── common_set_class_distribution.*
                ├── roc_comparison.*
                ├── precision_recall_comparison.*
                ├── fusion_confusion_matrix.*
                ├── probability_distributions.*
                ├── fusion_probability_by_true_class.*
                ├── prediction_correlation.*
                ├── regional_decision_agreement.*
                └── regional_vs_fusion_metrics.*
```

`*` grafiklerin hem PNG hem SVG olarak kaydedildiğini gösterir.